# 14-тарау. Генерация өнері: температура, top-k, top-p

*«Машина қалай үйренеді? 2-том» кітабына арналған блокнот. Әр ұяшықтың түсіндірмесі кітапта берілген.*

Ұяшықтарды жоғарыдан төмен қарай ретімен іске қосыңыз: **Shift + Enter**.

## 14.1. Ықтималдықтан мәтінге

In [ ]:
import re
import urllib.request
import torch

RAW = ("https://raw.githubusercontent.com/amirtemir1996n3-dotcom/"
       "mashina-qalai-uirenedi/main/")
for path in ["data/kitap.txt", "vol2/gpt.py", "vol2/gpt_kitap.pt"]:
    urllib.request.urlretrieve(RAW + path, path.split("/")[-1])
from gpt import GPT, GPTConfig

text = open("kitap.txt", encoding="utf-8").read()
chars = sorted(set(text))
stoi = {ch: i for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(chars[i] for i in ids)

block_size = 64
model = GPT(GPTConfig(vocab_size=len(chars), block_size=block_size))
model.load_state_dict(torch.load("gpt_kitap.pt"))
model.eval()
print("Модель жүктелді:", sum(p.numel() for p in model.parameters()))

## 14.2. Температура

In [ ]:
words = ["жақсы", "суық", "ыстық", "бұлтты", "ғажайып"]
logits = torch.tensor([2.0, 1.5, 1.0, 0.5, -0.5])
for T in [0.5, 1.0, 2.0]:
    probs = torch.softmax(logits / T, dim=0)
    print(f"T={T}:",
          "  ".join(f"{w} {p:.2f}" for w, p in zip(words, probs)))

## 14.3. Top-k және top-p

In [ ]:
def sample_next(logits, temperature=1.0, top_k=None, top_p=None):
    if temperature == 0:                         # ашкөз таңдау
        return int(torch.argmax(logits))
    probs = torch.softmax(logits / temperature, dim=-1)
    if top_k is not None:
        kth = torch.topk(probs, top_k).values[-1]
        probs = torch.where(probs >= kth, probs, 0.0)
    if top_p is not None:
        sorted_p, order = torch.sort(probs, descending=True)
        keep = torch.cumsum(sorted_p, dim=0) - sorted_p < top_p
        mask = torch.zeros_like(probs, dtype=torch.bool)
        mask[order[keep]] = True
        probs = torch.where(mask, probs, 0.0)
    probs = probs / probs.sum()
    return int(torch.multinomial(probs, 1))

probs = torch.softmax(logits, dim=0)
for name, kw in [("top-k=2", dict(top_k=2)),
                 ("top-p=0.8", dict(top_p=0.8))]:
    torch.manual_seed(0)
    picks = [words[sample_next(logits, **kw)] for _ in range(1000)]
    print(name, {w: picks.count(w) for w in words if picks.count(w)})

## 14.4. Стратегияларды модельде салыстырамыз

In [ ]:
@torch.no_grad()
def write(prompt, n=150, stop=None, **kw):
    ids = encode(prompt)
    for _ in range(n):
        x = torch.tensor([ids[-block_size:]])
        ids.append(sample_next(model(x)[0][0, -1], **kw))
        if stop is not None and chars[ids[-1]] == stop:
            break
    return decode(ids)

print(write("Машиналық оқыту", n=120, temperature=0))

In [ ]:
for kw in [dict(temperature=0.5), dict(temperature=1.0),
           dict(temperature=1.5), dict(temperature=1.0, top_p=0.9)]:
    torch.manual_seed(3)
    print(kw)
    print(write("Машиналық оқыту", n=100, stop=".", **kw))
    print()

In [ ]:
known = set(re.findall(r"\w+", text.lower()))

def quality(kw, n=2000):
    torch.manual_seed(0)
    out = write("Деректер", n=n, **kw)
    ws = re.findall(r"\w+", out.lower())
    real = sum(w in known for w in ws) / len(ws)
    unique = len(set(ws)) / len(ws)
    return real, unique

print("стратегия            шын сөздер  бірегей сөздер")
for name, kw in [("ашкөз", dict(temperature=0)),
                 ("T=0.5", dict(temperature=0.5)),
                 ("T=1.0", dict(temperature=1.0)),
                 ("T=1.0, top-p=0.9", dict(temperature=1.0, top_p=0.9)),
                 ("T=1.5", dict(temperature=1.5)),
                 ("T=1.5, top-p=0.9", dict(temperature=1.5, top_p=0.9))]:
    real, unique = quality(kw)
    print(f"{name:20s} {real:10.0%} {unique:15.0%}")